# Epoch-selection diagnostic — does compound-val select a different stopping point?

**This is NOT cheap like the bootstrap CI** — the per-epoch history was never logged
during the original training run, so answering "which epoch got selected, and why"
requires re-running training with epoch-by-epoch logging added. That's genuinely the
same per-seed GPU cost as before.

**Scoped down deliberately**: 2 seeds per variant (42, 43) instead of 5. The question
here is qualitative — does compound-val systematically stop earlier/later, with a
different loss/selection-F1 trajectory — not "what's the precise mean effect size."
2 seeds is enough to see the pattern; it is not enough to claim statistical significance,
and this notebook does not claim that.

**Requires Stage-4-v2's cached features/embeddings already computed** (`Xb`, `Xcat`,
`category_dims`, loaders) — run this in the same session/kernel as Stage 4 v2, after
Section 6 (model + trainer definitions), reusing `train_loader`, `sel_loader_original`,
`sel_loader_compound`, `test_loaders` from that notebook rather than rebuilding them.

## 1. Instrumented trainer — identical to `train_gated`, plus full epoch history

In [ ]:
def train_gated_with_history(train_loader, selection_loader, category_dims, seed,
                              epochs=15, lr=1e-3, weight_decay=0.01, patience=5):
    """Same architecture, same optimizer, same hyperparameters as train_gated -- the ONLY
    addition is recording (epoch, selection_f1, train_loss) every epoch, and which epoch
    the final checkpoint came from."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AttentionGatedHybridDetector(category_dims).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = get_cosine_schedule_with_warmup(opt, int(0.1*len(train_loader)*epochs), len(train_loader)*epochs)
    criterion = nn.CrossEntropyLoss()
    best_val_f1, best_state, best_epoch, epochs_no_improve = -1, None, -1, 0
    history = []
    for ep in range(1, epochs+1):
        model.train()
        total_loss = 0.0
        for be, cats, lb in train_loader:
            be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}; lb = lb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(be, cats), lb)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            total_loss += loss.item()
        sel_labels, sel_preds, _ = evaluate_gated(model, selection_loader)
        sel_f1 = f1_score(sel_labels, sel_preds, zero_division=0)
        avg_loss = total_loss / len(train_loader)
        history.append({'epoch': ep, 'train_loss': avg_loss, 'selection_f1': sel_f1})
        if sel_f1 > best_val_f1:
            best_val_f1, best_state, best_epoch, epochs_no_improve = sel_f1, {k: v.clone() for k, v in model.state_dict().items()}, ep, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
    model.load_state_dict(best_state)
    sel_labels, _, sel_probs = evaluate_gated(model, selection_loader)
    calibrated_t, _ = calibrate_threshold(sel_labels, sel_probs)
    return model, best_val_f1, calibrated_t, best_epoch, pd.DataFrame(history)


## 2. Run both variants, 2 seeds each, capture full epoch history

In [ ]:
DIAGNOSTIC_SEEDS = [42, 43]   # scoped down deliberately -- see note above

epoch_diagnostic_rows = []
history_records = []

for variant_name, selection_loader in [('Original (reddit-val)', sel_loader_original),
                                        ('Compound-val (reddit+dolly)', sel_loader_compound)]:
    for seed in DIAGNOSTIC_SEEDS:
        model, best_sel_f1, calibrated_t, best_epoch, history_df = train_gated_with_history(
            train_loader, selection_loader, category_dims, seed=seed)
        n_epochs_run = len(history_df)
        epoch_diagnostic_rows.append({
            'variant': variant_name, 'seed': seed, 'best_epoch': best_epoch,
            'total_epochs_run': n_epochs_run, 'best_selection_f1': best_sel_f1,
            'final_train_loss_at_best_epoch': history_df.loc[history_df.epoch == best_epoch, 'train_loss'].values[0],
        })
        history_df['variant'] = variant_name
        history_df['seed'] = seed
        history_records.append(history_df)
        print(f'[{variant_name}] seed {seed}: best_epoch={best_epoch}/{n_epochs_run}, '
              f'best_sel_f1={best_sel_f1:.4f}')
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

epoch_diagnostic_df = pd.DataFrame(epoch_diagnostic_rows)
full_history_df = pd.concat(history_records, ignore_index=True)

print()
print(epoch_diagnostic_df.to_string(index=False))
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
epoch_diagnostic_df.to_csv(f'{ARTIFACTS_DIR}/M4_epoch_selection_diagnostic.csv', index=False)
full_history_df.to_csv(f'{ARTIFACTS_DIR}/M4_epoch_selection_full_history.csv', index=False)


## 3. Interpretation guide

In [ ]:
print('Compare best_epoch between variants (same seed):')
pivot = epoch_diagnostic_df.pivot(index='seed', columns='variant', values='best_epoch')
print(pivot.to_string())
print()
print('If compound-val consistently selects a LATER epoch -- the harder signal keeps')
print('finding real improvement longer before plateauing/overfitting, i.e. it is not just')
print('stopping earlier out of noise, it is genuinely still learning something useful.')
print()
print('If compound-val consistently selects an EARLIER epoch -- the theory from before')
print('(harder signal triggers earlier, more conservative stopping) is supported directly.')
print()
print('If there is no consistent direction across the 2 seeds -- epoch selection is not')
print('the mechanism; the improvement likely comes from something else entirely (e.g. the')
print('gate weights themselves, or a genuinely different loss landscape from a harder target).')
print('This would mean the mechanism question needs the full 5-seed run to resolve, not 2.')
